In [1]:
import pandas as pd


In [29]:
df_NO2=pd.read_csv("stl_NO2_historic.csv")
df_PM25=pd.read_csv("stl_pm25_historic.csv")
df_weather=pd.read_csv("stl_weather_historic.csv")


In [30]:
df_NO2 = df_NO2.rename(columns={'date_local': 'date'})
df_PM25 = df_PM25.rename(columns={'date_local': 'date'})


In [ ]:
df_NO2['pollutant']='NO2'
df_PM25['pollutant']='PM25'


TypeError: cannot concatenate object of type '<class 'str'>'; only Series and DataFrame objs are valid

In [34]:
df_aq=pd.concat([df_NO2,df_PM25])


In [36]:
df_aq['site_number'].unique()


array([85, 94, 16,  7, 93])

In [37]:
df_aq.columns


Index(['date', 'site_number', 'sample_measurement', 'samp_dur_hrs',
       'pollutant'],
      dtype='str')

In [43]:
df_aq.groupby(['samp_dur_hrs','pollutant'])['site_number'].count()


samp_dur_hrs  pollutant
1             NO2          1095
              PM25         1825
24            PM25          115
Name: site_number, dtype: int64

In [ ]:
# PM25 is only measured once every three days
# only at location 85
# since we have full data for 1 hour data, for both pollutants, 
# and the scope of this project is to make a daily forcast I will drop it
df_aq.loc[df_aq['samp_dur_hrs']==24]['site_number'].unique()


array([85])

In [48]:
df_aq_clean=df_aq.loc[df_aq['samp_dur_hrs']==1]


In [50]:
df_aq_clean.drop(columns='samp_dur_hrs',inplace=True)


In [63]:
df_aq_final = df_aq_clean.pivot_table(
    index="date",
    columns="pollutant",
    values="sample_measurement",
    aggfunc="mean"
).reset_index()


In [64]:
df_aq_final.columns.name = None


In [ ]:
df_aq_final


,date,NO2,PM25
0,2025-01-01,7.030556,5.385000
1,2025-01-02,16.915278,9.868750
2,2025-01-03,4.408333,5.622083
3,2025-01-04,7.469444,4.595000
4,2025-01-05,6.011995,3.192905
...,...,...,...
360,2025-12-27,6.820833,8.487917
361,2025-12-28,4.397348,5.779167
362,2025-12-29,4.048611,3.660000
363,2025-12-30,9.562500,4.865952


In [71]:
df_weather=df_weather.set_index('date')
df_aq_final=df_aq_final.set_index('date')


In [76]:
df_full = df_weather.join(df_aq_final, how="outer")


In [77]:
df_full


,temperature_2m_max,precipitation_sum,precipitation_hours,uv_index_max,wind_speed_10m_max,shortwave_radiation_sum,NO2,PM25
date,,,,,,,,
2025-01-01,39.217266,0.000000,0.000000,2.699338,13.026237,7.356292,7.030556,5.385000
2025-01-02,43.549780,0.000662,0.006623,1.851987,10.110883,7.555298,16.915278,9.868750
2025-01-03,37.243824,0.000000,0.000000,2.747682,12.683156,8.675497,4.408333,5.622083
2025-01-04,30.915808,0.121192,0.251656,2.724172,8.053342,7.934370,7.469444,4.595000
2025-01-05,28.325480,32.121857,18.609272,0.668543,15.566028,2.632517,6.011995,3.192905
...,...,...,...,...,...,...,...,...
2025-12-27,60.747528,0.007947,0.019868,2.803311,11.757589,3.157218,6.820833,8.487917
2025-12-28,75.136800,0.028477,0.019868,2.183775,26.622955,4.699934,4.397348,5.779167
2025-12-29,24.975811,0.000000,0.000000,2.620530,24.417997,9.027019,4.048611,3.660000
